# 贝叶斯证据计算 - EDE vs EDE+PPF 模型比较

本 notebook 使用 MCEvidence 计算两个宇宙学模型的贝叶斯证据：
- **EDE 模型**: 早期暗能量 (Early Dark Energy)
- **EDE+PPF 模型**: 早期暗能量 + PPF (Parameterized Post-Friedmann)

数据集包括：
1. **CMBBAO / sCMBBAO**: CMB + BAO 观测数据
2. **CMBPANBAO / sCMBPANBAO**: CMB + Pantheon SN + BAO 数据
3. **DESY5 / sDESY5**: DES Year 5 弱引力透镜数据
4. **Union3 / sUnion3**: Union3 超新星数据

In [14]:
# 导入必要的库
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

# 添加 MCEvidence 路径
sys.path.insert(0, str(Path.cwd().parent))

# 导入 MCEvidence
from MCEvidence import MCEvidence

print("✓ 所有库导入成功！")
print(f"工作目录: {Path.cwd()}")

✓ 所有库导入成功！
工作目录: d:\My-project\Cosmology-MCMC\src\MCEvidence\notebook


In [15]:
# 配置参数 - 可以调整各种选项来控制证据计算的行为
# ==================== 数据路径配置 ====================
DATA_DIR = Path("d:/My-project/Cosmology-MCMC/EDE+PPF-model/data-analysis")

# ==================== 模型配置 ====================
# 注意：CMBBAO/CMBPANBAO/DESY5/Union3 是 EDE 模型
#      sCMBBAO/sCMBPANBAO/sDESY5/sUnion3 是 EDE+PPF 模型
MODELS = {
    'EDE': {
        'datasets': ['CMBBAO', 'CMBPANBAO', 'DESY5', 'Union3'],
        'description': 'Early Dark Energy (EDE) 模型'
    },
    'EDE+PPF': {
        'datasets': ['sCMBBAO', 'sCMBPANBAO', 'sDESY5', 'sUnion3'],
        'description': 'EDE + PPF 模型'
    }
}

# ==================== 数据集全称映射 ====================
# 用于在最终表格中显示完整的数据集名称
DATASET_FULL_NAMES = {
    'CMBBAO': 'CMB+DESI DR2 BAO +SH0ES',
    'sCMBBAO': 'CMB+DESI DR2 BAO +SH0ES',
    'CMBPANBAO': 'CMB+DESI DR2 BAO +Pantheon Plus+SH0ES',
    'sCMBPANBAO': 'CMB+DESI DR2 BAO +Pantheon Plus+SH0ES',
    'DESY5': 'CMB+DESI DR2 BAO +DESY5+SH0ES',
    'sDESY5': 'CMB+DESI DR2 BAO +DESY5+SH0ES',
    'Union3': 'CMB+DESI DR2 BAO +Union3+SH0ES',
    'sUnion3': 'CMB+DESI DR2 BAO +Union3+SH0ES'
}

# ==================== MCEvidence 计算参数 ====================
# kmax: k-最近邻算法的最大 k 值
#       - 计算 k=1,2,3,...,kmax-1 的贝叶斯证据
#       - k=2 通常最稳定可靠
#       - 推荐值: 5 (计算 k=1,2,3,4)
KMAX = 5

# burnlen: Burn-in 比例
#          - 丢弃链开始的这部分样本（热身阶段）
#          - 0.3 表示丢弃前 30% 的样本
#          - 推荐值: 0.3 (30%)
BURNLEN = 0.5

# thinlen: 稀疏化（thinning）间隔
#          - 1 表示不稀疏化（使用所有样本）
#          - >1 表示每隔 thinlen 个样本取一个
#          - 通常设为 1，除非链有很强的自相关
THINLEN = 1

# verbose: 输出详细程度
#          - 0: 静默模式
#          - 1: 基本信息
#          - 2: 详细调试信息
VERBOSE = 0

# split: 是否使用交叉证据（cross-evidence）
#        - False: 使用全部样本计算（更快，推荐）
#        - True: 分割样本计算交叉证据（更保守，较慢）
SPLIT = False

# ==================== 参数选择模式 ====================
# PARAM_MODE 控制使用哪些参数计算证据
#
# 选项1: 'cosmo_only' - 仅使用宇宙学参数
#        包括: H0, logA, ns, ombh2, omch2, tau, fde_zc, log10_ac, theta_i, w, wa
#        优点: 聚焦于理论参数，计算快
#        缺点: 忽略系统误差的不确定性
#
# 选项2: 'with_nuisance' - 包括 nuisance 参数
#        额外包括: A_planck, amp_143, amp_217, amp_143x217, n_143, n_217, n_143x217, calTE, calEE
#        优点: 更保守的估计，考虑系统误差
#        缺点: 维度更高，计算稍慢
#
# 选项3: None - 使用 YAML 中所有采样参数（默认）
#        自动从 Cobaya yaml 文件读取
PARAM_MODE = 'cosmo_only'  # 推荐: 用于模型比较

print("配置完成！")
print(f"数据目录: {DATA_DIR}")
print(f"参数模式: {PARAM_MODE}")
print(f"kmax={KMAX}, burnlen={BURNLEN}, thinlen={THINLEN}")

配置完成！
数据目录: d:\My-project\Cosmology-MCMC\EDE+PPF-model\data-analysis
参数模式: cosmo_only
kmax=5, burnlen=0.5, thinlen=1


In [16]:
# 定义计算函数 - 计算单个数据集的贝叶斯证据
def calculate_evidence(chain_path, dataset_name, model_name, param_mode=None):
    """
    计算单个数据集的贝叶斯证据

    参数:
    -----
    chain_path : Path
        链文件所在目录
    dataset_name : str
        数据集名称
    model_name : str
        模型名称
    param_mode : str or None
        参数选择模式 ('cosmo_only', 'with_nuisance', None)

    返回:
    -----
    dict : 包含证据值和相关信息的字典
    """
    print(f"\n{'='*70}")
    print(f"计算: {model_name} - {dataset_name}")
    print(f"{'='*70}")

    try:
        # 初始化 MCEvidence
        mce = MCEvidence(
            str(chain_path / dataset_name),
            kmax=KMAX,
            burnlen=BURNLEN,
            thinlen=THINLEN,
            verbose=VERBOSE,
            split=SPLIT
        )

        # 根据参数模式设置维度
        if param_mode == 'cosmo_only' and hasattr(mce.gd, 'cobaya_param_info'):
            param_info = mce.gd.cobaya_param_info
            if param_info:
                mce.ndim = param_info['n_cosmo']
                print(f"✓ 使用 {mce.ndim} 个宇宙学参数")
                print(f"  参数: {', '.join(param_info['cosmo_params'])}")
        elif param_mode == 'with_nuisance' and hasattr(mce.gd, 'cobaya_param_info'):
            param_info = mce.gd.cobaya_param_info
            if param_info:
                mce.ndim = param_info['n_sampled']
                print(f"✓ 使用 {mce.ndim} 个采样参数（宇宙学 + nuisance）")

        # 计算证据
        print("\n运行证据计算...")
        ln_evidence = mce.evidence()

        # 提取结果
        result = {
            'model': model_name,
            'dataset': dataset_name,
            'n_samples': mce.gd.get_shape()[0],
            'ndim': mce.ndim,
            'ln_Z_k1': ln_evidence[0],
            'ln_Z_k2': ln_evidence[1] if len(ln_evidence) > 1 else ln_evidence[0],
            'ln_Z_k3': ln_evidence[2] if len(ln_evidence) > 2 else ln_evidence[0],
            'ln_Z_k4': ln_evidence[3] if len(ln_evidence) > 3 else ln_evidence[0],
            'status': 'SUCCESS'
        }

        # 使用 k=2 作为最佳值
        result['ln_Z_best'] = result['ln_Z_k2']

        print(f"\n{'─'*70}")
        print(f"结果:")
        print(f"{'─'*70}")
        for k_idx in range(len(ln_evidence)):
            print(f"  ln(Z) [k={k_idx+1}] = {ln_evidence[k_idx]:.3f}")
        print(f"{'─'*70}")
        print(f"✓ 计算成功！")

        return result

    except Exception as e:
        print(f"✗ 计算失败: {e}")
        import traceback
        traceback.print_exc()
        return {
            'model': model_name,
            'dataset': dataset_name,
            'status': 'FAILED',
            'error': str(e)
        }

print("✓ 函数定义完成！")

✓ 函数定义完成！


In [17]:
# 批量计算所有数据集 - 计算 8 个数据集（4 个 EDE + 4 个 EDE+PPF）的贝叶斯证据
# 存储所有结果
all_results = []

# 遍历所有模型和数据集
for model_name, model_info in MODELS.items():
    print(f"\n\n{'#'*70}")
    print(f"# 模型: {model_name} - {model_info['description']}")
    print(f"{'#'*70}")

    for dataset in model_info['datasets']:
        chain_path = DATA_DIR / dataset

        # 检查数据是否存在
        if not chain_path.exists():
            print(f"\n✗ 数据不存在: {chain_path}")
            continue

        # 计算证据
        result = calculate_evidence(chain_path, dataset, model_name, PARAM_MODE)
        all_results.append(result)

# 转换为 DataFrame
results_df = pd.DataFrame(all_results)

print(f"\n\n{'='*70}")
print("所有计算完成！")
print(f"{'='*70}")
print(f"成功: {len(results_df[results_df['status']=='SUCCESS'])}/{len(results_df)}")
print(f"失败: {len(results_df[results_df['status']=='FAILED'])}/{len(results_df)}")



######################################################################
# 模型: EDE - Early Dark Energy (EDE) 模型
######################################################################

计算: EDE - CMBBAO
✓ 使用 9 个宇宙学参数
  参数: H0, logA, ns, ombh2, omch2, tau, fde_zc, log10_ac, theta_i

运行证据计算...
✓ 使用 9 个宇宙学参数
  参数: H0, logA, ns, ombh2, omch2, tau, fde_zc, log10_ac, theta_i

运行证据计算...

──────────────────────────────────────────────────────────────────────
结果:
──────────────────────────────────────────────────────────────────────
  ln(Z) [k=1] = -5537.365
  ln(Z) [k=2] = -5536.823
  ln(Z) [k=3] = -5536.507
  ln(Z) [k=4] = -5536.265
──────────────────────────────────────────────────────────────────────
✓ 计算成功！

计算: EDE - CMBPANBAO

──────────────────────────────────────────────────────────────────────
结果:
──────────────────────────────────────────────────────────────────────
  ln(Z) [k=1] = -5537.365
  ln(Z) [k=2] = -5536.823
  ln(Z) [k=3] = -5536.507
  ln(Z) [k=4] = -5536.265
─────────────────

In [18]:
# 查看原始结果 - 只显示成功的结果
# 只显示成功的结果
successful_df = results_df[results_df['status'] == 'SUCCESS'].copy()

# 显示主要列
display_cols = ['model', 'dataset', 'n_samples', 'ndim', 'ln_Z_best']
print("\n原始计算结果:")
print("="*80)
display(successful_df[display_cols].style.format({
    'ln_Z_best': '{:.3f}',
    'n_samples': '{:,}'
}))


原始计算结果:


,model,dataset,n_samples,ndim,ln_Z_best
0,EDE,CMBBAO,"38,850",9,-5536.823
1,EDE,CMBPANBAO,"60,000",9,-6240.040
2,EDE,DESY5,"60,000",9,-6362.394
3,EDE,Union3,"60,000",9,-5551.282
4,EDE+PPF,sCMBBAO,"23,351",11,-5538.663
5,EDE+PPF,sCMBPANBAO,"9,014",11,-6243.322
6,EDE+PPF,sDESY5,"9,014",11,-6362.613
7,EDE+PPF,sUnion3,"9,014",11,-5554.144


In [20]:
# 创建比较表格 - 横轴（列）: 四组数据集全称，纵轴（行）: 两个模型，最后一行: Δln(Z) = ln(Z_EDE+PPF) - ln(Z_EDE)
# 创建数据集名称映射（去掉 's' 前缀统一命名）
dataset_mapping = {
    'CMBBAO': 'CMBBAO',
    'sCMBBAO': 'CMBBAO',
    'CMBPANBAO': 'CMBPANBAO',
    'sCMBPANBAO': 'CMBPANBAO',
    'DESY5': 'DESY5',
    'sDESY5': 'DESY5',
    'Union3': 'Union3',
    'sUnion3': 'Union3'
}

# 添加统一的数据集名称列
successful_df['dataset_clean'] = successful_df['dataset'].map(dataset_mapping)

# 创建透视表：行是模型，列是数据集
comparison_table = successful_df.pivot(
    index='model',
    columns='dataset_clean',
    values='ln_Z_best'
)

# 确保列的顺序
column_order = ['CMBBAO', 'CMBPANBAO', 'Union3', 'DESY5']
comparison_table = comparison_table[column_order]

# 计算差值行：EDE+PPF - EDE
if 'EDE' in comparison_table.index and 'EDE+PPF' in comparison_table.index:
    delta_row = comparison_table.loc['EDE+PPF'] - comparison_table.loc['EDE']
    delta_row.name = 'Δln(Z) = EDE+PPF - EDE'

    # 添加到表格
    comparison_table = pd.concat([comparison_table, delta_row.to_frame().T])

# 将列名改为全称
comparison_table.columns = [DATASET_FULL_NAMES[col] for col in comparison_table.columns]

print("\n模型比较表格:")
print("="*80)
print("ln(Z) 值对比")
print("="*80)
display(comparison_table.style.format('{:.3f}').set_properties(**{
    'text-align': 'center',
    'font-weight': 'bold'
}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'center'), ('font-weight', 'bold')]},
    {'selector': 'td', 'props': [('text-align', 'center')]}
]))


模型比较表格:
ln(Z) 值对比


,CMB+DESI DR2 BAO +SH0ES,CMB+DESI DR2 BAO +Pantheon Plus+SH0ES,CMB+DESI DR2 BAO +Union3+SH0ES,CMB+DESI DR2 BAO +DESY5+SH0ES
EDE,-5536.823,-6240.040,-5551.282,-6362.394
EDE+PPF,-5538.663,-6243.322,-5554.144,-6362.613
Δln(Z) = EDE+PPF - EDE,-1.840,-3.282,-2.863,-0.218


In [21]:
# 贝叶斯因子解释 - Δln(Z) 的解释标准
# Δln(Z) > 5: 非常强的证据支持 EDE+PPF 模型
# 2.5 < Δln(Z) < 5: 强证据支持 EDE+PPF 模型
# 1 < Δln(Z) < 2.5: 中等证据支持 EDE+PPF 模型
# -1 < Δln(Z) < 1: 无结论（两个模型几乎相等）
# -2.5 < Δln(Z) < -1: 中等证据支持 EDE 模型
# -5 < Δln(Z) < -2.5: 强证据支持 EDE 模型
# Δln(Z) < -5: 非常强的证据支持 EDE 模型
# 创建解释表格
interpretations = []

for dataset in column_order:
    if 'Δln(Z) = EDE+PPF - EDE' in comparison_table.index:
        delta = comparison_table.loc['Δln(Z) = EDE+PPF - EDE', DATASET_FULL_NAMES[dataset]]

        # 判断倾向
        if delta > 5:
            interpretation = "非常强支持 EDE+PPF"
            color = 'darkgreen'
        elif delta > 2.5:
            interpretation = "强支持 EDE+PPF"
            color = 'green'
        elif delta > 1:
            interpretation = "中等支持 EDE+PPF"
            color = 'lightgreen'
        elif delta > -1:
            interpretation = "无结论"
            color = 'gray'
        elif delta > -2.5:
            interpretation = "中等支持 EDE"
            color = 'lightcoral'
        elif delta > -5:
            interpretation = "强支持 EDE"
            color = 'red'
        else:
            interpretation = "非常强支持 EDE"
            color = 'darkred'

        # 计算贝叶斯因子
        BF = np.exp(delta)

        interpretations.append({
            '数据集': DATASET_FULL_NAMES[dataset],
            'Δln(Z)': delta,
            '贝叶斯因子 (B)': BF,
            '解释': interpretation
        })

interp_df = pd.DataFrame(interpretations)

print("\n贝叶斯因子详细解释:")
print("="*80)
display(interp_df.style.format({
    'Δln(Z)': '{:.3f}',
    '贝叶斯因子 (B)': '{:.2e}'
}).set_properties(**{
    'text-align': 'center'
}))


贝叶斯因子详细解释:


,数据集,Δln(Z),贝叶斯因子 (B),解释
0,CMB+DESI DR2 BAO +SH0ES,-1.840,1.59e-01,中等支持 EDE
1,CMB+DESI DR2 BAO +Pantheon Plus+SH0ES,-3.282,3.75e-02,强支持 EDE
2,CMB+DESI DR2 BAO +Union3+SH0ES,-2.863,5.71e-02,强支持 EDE
3,CMB+DESI DR2 BAO +DESY5+SH0ES,-0.218,8.04e-01,无结论
